<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/golf/01_logistic_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Golf putting 1 — Logistic baseline

Start with the conventional statistical model. The goal is not to make logistic regression “win”; it is to establish a baseline and practice the workflow before introducing golf-specific mechanism.

## Setup

The notebook pins the current PyMC 6 / ArviZ 1.x stack used in the course.

In [ ]:
%pip install -q pymc "arviz-plots[matplotlib]" arviz-stats

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

DATA_BASE = "https://raw.githubusercontent.com/opherdonchin/BayesShortCourse/main/golf/data"
golf = pd.read_csv(f"{DATA_BASE}/berry_1996_putting.csv")
golf["rate"] = golf["made"] / golf["attempts"]

BALL_RADIUS_FT = (1.68 / 2) / 12
CUP_RADIUS_FT = (4.25 / 2) / 12

golf

In [ ]:
def plot_data(data, title=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(data["distance_ft"], data["rate"], s=35)
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        ylim=(-0.03, 1.03),
        title=title,
    )
    return ax

def plot_predictive_rate(dt, group, data, var_name, title, prob=0.90, show_observed=True):
    draws = dt[group][var_name]
    median = draws.median(dim=("chain", "draw"))
    interval = draws.azstats.hdi(prob=prob)
    x = data["distance_ft"].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(
        x,
        interval.sel(ci_bound="lower"),
        interval.sel(ci_bound="upper"),
        alpha=0.22,
        label=f"{prob:.0%} HDI",
    )
    ax.plot(x, median, label="Predictive median")
    if show_observed:
        ax.scatter(x, data["rate"], s=35, label="Observed")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        ylim=(-0.05, 1.05),
        title=title,
    )
    ax.legend()
    return ax

def plot_latent_fit(idata, data, var_name="p_base", title="Underlying fitted relationship"):
    p = idata["posterior"][var_name]
    median = p.median(dim=("chain", "draw"))
    interval = p.azstats.hdi(prob=0.90)
    x = data["distance_ft"].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(
        x,
        interval.sel(ci_bound="lower"),
        interval.sel(ci_bound="upper"),
        alpha=0.22,
        label="90% HDI",
    )
    ax.plot(x, median, label="Posterior median")
    ax.scatter(x, data["rate"], s=35, label="Observed")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        ylim=(-0.03, 1.03),
        title=title,
    )
    ax.legend()
    return ax

def plot_residuals(idata, data, var_name="p_base", title="Residuals"):
    fitted = idata["posterior"][var_name].median(dim=("chain", "draw"))
    residual = data["rate"].to_numpy() - fitted.to_numpy()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.plot(data["distance_ft"], residual, marker="o")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Observed − fitted probability",
        title=title,
    )
    return ax

## Model

$y_j\sim\operatorname{Binomial}(n_j,p_j),\qquad \operatorname{logit}(p_j)=\alpha+\beta x_j.$

This is deliberately generic: distance enters only through a straight line on the log-odds scale.

In [ ]:
coords = {"obs_id": golf["distance_ft"].to_numpy()}

with pm.Model(coords=coords) as model:
    distance = pm.Data("distance", golf["distance_ft"].to_numpy(), dims="obs_id")
    attempts = pm.Data("attempts", golf["attempts"].to_numpy(), dims="obs_id")
    made_data = pm.Data("made_data", golf["made"].to_numpy(), dims="obs_id")

    intercept = pm.Normal("intercept", mu=0, sigma=3)
    slope = pm.Normal("slope", mu=0, sigma=0.5)
    p_base = pm.Deterministic(
        "p_base",
        pm.math.sigmoid(intercept + slope * distance),
        dims="obs_id",
    )
    p = pm.Deterministic("p", p_base, dims="obs_id")

    made = pm.Binomial(
        "made",
        n=attempts,
        p=p,
        observed=made_data,
        dims="obs_id",
    )
    pm.Deterministic("made_rate", made / attempts, dims="obs_id")

## Prior predictive check

Check what the model can generate **before conditioning on the observed successes**. The design variables (distance and number of attempts) are fixed; the outcomes are simulated.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=500, random_seed=RANDOM_SEED)

In [ ]:
plot_predictive_rate(prior, "prior_predictive", golf, "made_rate", "Prior predictive", show_observed=False);

## Fit and diagnose

Do not interpret the scientific fit until the sampler diagnostics are acceptable.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.9,
        nuts_sampler="pymc",
        random_seed=RANDOM_SEED,
    )

print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))
azs.summary(
    idata,
    var_names=['intercept', 'slope'],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(idata, var_names=['intercept', 'slope']);

## Posterior predictive check

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
plot_predictive_rate(idata, "posterior_predictive", golf, "made_rate", "Posterior predictive check");

In [ ]:
plot_latent_fit(idata, golf, var_name="p_base", title="Logistic relationship");
plot_residuals(idata, golf, var_name="p_base", title="Logistic residuals");

## Decision: revise the model

The logistic curve is a useful baseline, but it has no representation of the geometry of putting. Even if its in-sample fit were tolerable, its extrapolation is driven by the arbitrary logit-linear form. **Next notebook:** replace the generic curve with a mechanism-based angle model.